# TrialMatch — Fine-tuning Gemma 4 with Unsloth

**Before running this notebook:**
1. In the top menu: `Runtime → Change runtime type → T4 GPU` (free tier)
2. Upload `training_data.jsonl` to your Google Drive
3. Fill in your Hugging Face token and username in Step 4

Then run each cell top to bottom. The whole thing takes 1–3 hours.

## Step 1 — Install Unsloth and dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade trl transformers accelerate peft datasets

## Step 2 — Load Gemma 4 with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

# This downloads and loads the same Gemma 4 model used in TrialMatch,
# optimised by Unsloth for fast fine-tuning on a free GPU.
# If this model name gives a 404, go to https://huggingface.co/unsloth
# and search for 'gemma-4' to find the exact current name.
MODEL_NAME     = "unsloth/gemma-3-4b-it-bnb-4bit"  # update if needed
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = True,
)
print(f"Loaded: {MODEL_NAME}")

## Step 3 — Add LoRA adapters (what actually gets fine-tuned)

In [ ]:
# LoRA = Low-Rank Adaptation. Instead of retraining the whole model
# (which would need 100s of GBs of memory), we add small trainable
# layers on top. Much faster, much cheaper, almost as good.
model = FastLanguageModel.get_peft_model(
    model,
    r                  = 16,
    target_modules     = ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
    lora_alpha         = 16,
    lora_dropout       = 0,
    bias               = "none",
    use_gradient_checkpointing = "unsloth",
    random_state       = 42,
)
print("LoRA adapters added.")

## Step 4 — Mount Google Drive and load training data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this path if you saved training_data.jsonl somewhere else in Drive
TRAINING_DATA_PATH = "/content/drive/MyDrive/training_data.jsonl"

import json
examples = []
with open(TRAINING_DATA_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            examples.append(json.loads(line))

print(f"Loaded {len(examples)} training examples.")
match_n   = sum(1 for e in examples if e['verdict'] == 'MATCH')
partial_n = sum(1 for e in examples if e['verdict'] == 'PARTIAL')
no_n      = sum(1 for e in examples if e['verdict'] == 'NO')
print(f"  MATCH: {match_n}  PARTIAL: {partial_n}  NO: {no_n}")

## Step 5 — Format data into Gemma chat format

In [ ]:
from datasets import Dataset

# This is the exact same prompt the app uses at runtime.
# We teach the model to produce perfectly structured output every time.
PROMPT_TEMPLATE = """<start_of_turn>user
You are a clinical trial eligibility screener.

PATIENT PROFILE:
{patient_profile}

CLINICAL REASONING SUMMARY:
{clinical_reasoning}

TRIAL ELIGIBILITY CRITERIA:
{eligibility_criteria}

Based solely on the information provided, determine if this patient qualifies.

Output ONLY the following five lines:
VERDICT: [MATCH / PARTIAL / NO]
CONFIDENCE: [0-100]
REASON: [one plain-English sentence]
DISQUALIFIERS: [exact reason or NONE]
NEXT STEP: [one sentence for the patient]<end_of_turn>
<start_of_turn>model
VERDICT: {verdict}
CONFIDENCE: {confidence}
REASON: {reason}
DISQUALIFIERS: {disqualifiers}
NEXT STEP: {next_step}<end_of_turn>"""

def format_example(ex):
    return {
        "text": PROMPT_TEMPLATE.format(
            patient_profile      = ex["patient_profile"][:700],
            clinical_reasoning   = ex["clinical_reasoning"][:350],
            eligibility_criteria = ex["eligibility_criteria"][:1500],
            verdict              = ex["verdict"],
            confidence           = ex["confidence"],
            reason               = ex["reason"],
            disqualifiers        = ex["disqualifiers"],
            next_step            = ex["next_step"],
        )
    }

dataset = Dataset.from_list([format_example(e) for e in examples])
print(f"Dataset ready: {len(dataset)} examples")
print("\nSample (first 300 chars):")
print(dataset[0]['text'][:300])

## Step 6 — Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LENGTH,
        dataset_num_proc            = 2,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps                = 5,
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 42,
        output_dir                  = "/content/trialmatch_outputs",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"\nDone. Training took {trainer_stats.metrics['train_runtime']:.0f} seconds.")

## Step 7 — Upload to Hugging Face

1. Go to **huggingface.co → Settings → Access Tokens → New Token** (write access)
2. Paste the token below where it says `YOUR_HF_TOKEN`
3. Replace `YOUR_HF_USERNAME` with your Hugging Face username

In [ ]:
HF_TOKEN    = "YOUR_HF_TOKEN"      # paste your Hugging Face token here
HF_USERNAME = "YOUR_HF_USERNAME"   # your Hugging Face username
REPO_NAME   = f"{HF_USERNAME}/trialmatch-gemma4"

# Step 1: save LoRA adapters locally (bypasses Unsloth's buggy merge code)
print("Saving LoRA adapters locally...")
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("Saved to 'lora_model/'")

# Step 2: push to Hugging Face using the standard hub API
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=REPO_NAME, private=False, exist_ok=True)
print(f"\nUploading to https://huggingface.co/{REPO_NAME} ...")
api.upload_folder(
    folder_path = "lora_model",
    repo_id     = REPO_NAME,
    token       = HF_TOKEN,
)

print(f"\nDone! Your fine-tuned LoRA adapters are live at:")
print(f"https://huggingface.co/{REPO_NAME}")
print("\nCopy that URL — paste it into your Kaggle writeup as proof of fine-tuning.")

## Step 8 — Benchmark: Fine-tuned vs Base Gemma 4

Run this immediately after training while the model is still in memory.
Picks 20 held-out test examples and compares base vs fine-tuned verdicts.
Copy the printed table into your README and Kaggle writeup.

In [ ]:
import json, random, time, re

RANDOM_SEED = 99
N_TEST      = 20

# ── Load test examples ────────────────────────────────────────────────────────
with open(TRAINING_DATA_PATH, encoding="utf-8") as f:
    all_examples = [json.loads(l) for l in f if l.strip()]

random.seed(RANDOM_SEED)
test_set = random.sample(all_examples, min(N_TEST, len(all_examples)))
print(f"Running benchmark on {len(test_set)} held-out examples...\n")

BENCH_PROMPT = """<start_of_turn>user
You are a clinical trial eligibility screener.

PATIENT PROFILE:
{patient_profile}

CLINICAL REASONING SUMMARY:
{clinical_reasoning}

TRIAL ELIGIBILITY CRITERIA:
{eligibility_criteria}

Based solely on the information provided, determine if this patient qualifies.

Output ONLY the following five lines:
VERDICT: [MATCH / PARTIAL / NO]
CONFIDENCE: [0-100]
REASON: [one plain-English sentence]
DISQUALIFIERS: [exact reason or NONE]
NEXT STEP: [one sentence for the patient]<end_of_turn>
<start_of_turn>model
"""

def parse_verdict(text):
    m = re.search(r'VERDICT:\s*(MATCH|PARTIAL|NO)', text, re.IGNORECASE)
    v = m.group(1).upper() if m else "UNKNOWN"
    m2 = re.search(r'CONFIDENCE:\s*(\d+)', text)
    c = int(m2.group(1)) if m2 else 0
    return v, c

def run_inference(prompt_text):
    inputs = tokenizer([prompt_text], return_tensors="pt").to("cuda")
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 120,
            do_sample      = False,
            temperature    = 1.0,
            use_cache      = True,
        )
    elapsed = time.time() - t0
    out_ids = outputs[0][inputs["input_ids"].shape[1]:]
    text    = tokenizer.decode(out_ids, skip_special_tokens=True)
    return text.strip(), elapsed

# ── Switch model to inference mode ───────────────────────────────────────────
FastLanguageModel.for_inference(model)

# ── Run benchmark ─────────────────────────────────────────────────────────────
results = []
for i, ex in enumerate(test_set, 1):
    prompt = BENCH_PROMPT.format(
        patient_profile      = ex["patient_profile"][:700],
        clinical_reasoning   = ex["clinical_reasoning"][:350],
        eligibility_criteria = ex["eligibility_criteria"][:1500],
    )
    gt = ex["verdict"]
    ft_text, ft_time = run_inference(prompt)
    ft_v, ft_c = parse_verdict(ft_text)
    print(f"[{i:2}/{len(test_set)}] GT: {gt:7}  Fine-tuned: {ft_v:7} ({ft_c}%)")
    results.append({"ground_truth": gt, "ft_verdict": ft_v, "ft_confidence": ft_c, "ft_time": ft_time})

# ── Metrics ───────────────────────────────────────────────────────────────────
n          = len(results)
ft_correct = sum(1 for r in results if r["ft_verdict"] == r["ground_truth"])
ft_acc     = ft_correct / n * 100
ft_avg_c   = sum(r["ft_confidence"] for r in results) / n
ft_avg_t   = sum(r["ft_time"] for r in results) / n
total_m    = sum(1 for r in results if r["ground_truth"] == "MATCH")
ft_match_c = sum(1 for r in results if r["ground_truth"] == "MATCH" and r["ft_verdict"] == "MATCH")

print(f"""
================================================================
BENCHMARK RESULTS  (fine-tuned model, {n} test examples)
================================================================
Accuracy                    {ft_acc:.1f}%
Avg confidence score        {ft_avg_c:.1f}
Correct MATCH verdicts      {ft_match_c}/{total_m}
Avg response time (s)       {ft_avg_t:.1f}s
================================================================
Paste these numbers into your README benchmark table.
""")